# Working with multiple datasets, using joins and merges

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum, avg, round, broadcast, desc, coalesce, lit
)

In [2]:
spark = SparkSession.builder.appName("BankingJoins").getOrCreate()

In [3]:
### Loading the datasets

In [5]:
customers     = spark.read.csv("../datasets/banking/customers.csv",     header=True, inferSchema=True)
transactions  = spark.read.csv("../datasets/banking/transactions.csv",  header=True, inferSchema=True)
loans         = spark.read.csv("../datasets/banking/loans.csv",         header=True, inferSchema=True)
branches      = spark.read.csv("../datasets/banking/branches.csv",      header=True, inferSchema=True)
credit_scores = spark.read.csv("../datasets/banking/credit_scores.csv", header=True, inferSchema=True)

print("================== Customer data ===================")
customers.show()
print("================== Transaction data ===================")
transactions.show()
print("================== Loan data ===================")
loans.show()
print("================== Branches data ===================")
branches.show()
print("================== Credit Scores data ===================")
credit_scores.show()

================== Customer data ===================
+-----------+-----------+---+---------+--------+-------------+----------+
|customer_id|       name|age|     city| segment|annual_income| join_date|
+-----------+-----------+---+---------+--------+-------------+----------+
|       C101|Arun Sharma| 34|   Mumbai| Premium|       850000|2019-03-15|
|       C102|  Priya Sen| 28|    Delhi|Standard|      1200000|2020-07-22|
|       C103|  Rahul Das| 45|  Kolkata|Standard|       450000|2018-11-10|
|       C104| Meena Iyer| 52|  Chennai| Premium|       950000|2017-05-30|
|       C105| Suresh Roy| 23|     Pune|   Basic|       320000|2023-01-14|
|       C106|Fatima Khan| 39|Hyderabad| Premium|       780000|2021-09-05|
|       C107|Vikram Nair| 31|Bangalore|Standard|      1500000|2020-02-28|
|       C108|Deepa Menon| 44|    Kochi|   Basic|       290000|2022-06-18|
+-----------+-----------+---+---------+--------+-------------+----------+

================== Transaction data ===================
+-

### Syntax

- if column name is same in both the dataframe 

In [7]:
result = transactions.join(customers, on="customer_id", how="inner")
result.show()

+-----------+--------------+----------+------------------+----------------+----------------+---------+-----------+---+---------+--------+-------------+----------+
|customer_id|transaction_id|account_id|transaction_amount|transaction_type|transaction_date|branch_id|       name|age|     city| segment|annual_income| join_date|
+-----------+--------------+----------+------------------+----------------+----------------+---------+-----------+---+---------+--------+-------------+----------+
|       C101|          T001|     A1001|            5000.0|           Debit|      2024-01-15|      B01|Arun Sharma| 34|   Mumbai| Premium|       850000|2019-03-15|
|       C102|          T002|     A1002|           15000.0|          Credit|      2024-01-16|      B02|  Priya Sen| 28|    Delhi|Standard|      1200000|2020-07-22|
|       C103|          T003|     A1003|            2000.0|           Debit|      2024-01-17|      B03|  Rahul Das| 45|  Kolkata|Standard|       450000|2018-11-10|
|       C101|         

- when the column name differs we need to mention the joining condition explicitly

In [9]:
result = transactions.join(customers,
            transactions.customer_id == customers.customer_id,
            "inner")
result.show()

+--------------+-----------+----------+------------------+----------------+----------------+---------+-----------+-----------+---+---------+--------+-------------+----------+
|transaction_id|customer_id|account_id|transaction_amount|transaction_type|transaction_date|branch_id|customer_id|       name|age|     city| segment|annual_income| join_date|
+--------------+-----------+----------+------------------+----------------+----------------+---------+-----------+-----------+---+---------+--------+-------------+----------+
|          T001|       C101|     A1001|            5000.0|           Debit|      2024-01-15|      B01|       C101|Arun Sharma| 34|   Mumbai| Premium|       850000|2019-03-15|
|          T002|       C102|     A1002|           15000.0|          Credit|      2024-01-16|      B02|       C102|  Priya Sen| 28|    Delhi|Standard|      1200000|2020-07-22|
|          T003|       C103|     A1003|            2000.0|           Debit|      2024-01-17|      B03|       C103|  Rahul Das

### Inner join: we keep only the matching records

In [11]:
txn_with_customer = transactions.join(
    customers,
    on="customer_id",
    how="inner"
)
txn_with_customer.show(truncate=False)

print(f"Transactions: {transactions.count()}")
print(f"Customers:    {customers.count()}")
print(f"Inner join:   {txn_with_customer.count()}")


+-----------+--------------+----------+------------------+----------------+----------------+---------+-----------+---+---------+--------+-------------+----------+
|customer_id|transaction_id|account_id|transaction_amount|transaction_type|transaction_date|branch_id|name       |age|city     |segment |annual_income|join_date |
+-----------+--------------+----------+------------------+----------------+----------------+---------+-----------+---+---------+--------+-------------+----------+
|C101       |T001          |A1001     |5000.0            |Debit           |2024-01-15      |B01      |Arun Sharma|34 |Mumbai   |Premium |850000       |2019-03-15|
|C102       |T002          |A1002     |15000.0           |Credit          |2024-01-16      |B02      |Priya Sen  |28 |Delhi    |Standard|1200000      |2020-07-22|
|C103       |T003          |A1003     |2000.0            |Debit           |2024-01-17      |B03      |Rahul Das  |45 |Kolkata  |Standard|450000       |2018-11-10|
|C101       |T004     

## Left join : When we need all the records of the left df

In [13]:
# Left join: all customers, with their transactions where available
all_customers_txns = customers.join(
    transactions,
    on="customer_id",
    how="left"
)

all_customers_txns.show(truncate=False)
print(f"Result rows: {all_customers_txns.count()}")


+-----------+-----------+---+---------+--------+-------------+----------+--------------+----------+------------------+----------------+----------------+---------+
|customer_id|name       |age|city     |segment |annual_income|join_date |transaction_id|account_id|transaction_amount|transaction_type|transaction_date|branch_id|
+-----------+-----------+---+---------+--------+-------------+----------+--------------+----------+------------------+----------------+----------------+---------+
|C101       |Arun Sharma|34 |Mumbai   |Premium |850000       |2019-03-15|T004          |A1001     |3000.0            |Debit           |2024-01-18      |B01      |
|C101       |Arun Sharma|34 |Mumbai   |Premium |850000       |2019-03-15|T001          |A1001     |5000.0            |Debit           |2024-01-15      |B01      |
|C102       |Priya Sen  |28 |Delhi    |Standard|1200000      |2020-07-22|T007          |A1002     |8000.0            |Debit           |2024-01-21      |B02      |
|C102       |Priya Sen

## Find the customer with no transaction

In [15]:
inactive_customers = customers.join(
    transactions,
    on="customer_id",
    how="left"
).filter(col("transaction_id").isNull())

inactive_customers.select("customer_id", "name", "city", "segment").show()


+-----------+-----------+-----+-------+
|customer_id|       name| city|segment|
+-----------+-----------+-----+-------+
|       C108|Deepa Menon|Kochi|  Basic|
+-----------+-----------+-----+-------+



- similar for right join, it keeps all the records of the right df

## Full outer join- keep all the records from both the df

In [16]:
full_view = customers.join(
    transactions,
    on="customer_id",
    how="outer"     # also valid: how="full" or how="full_outer"
)

full_view.show(truncate=False)
print(f"Result rows: {full_view.count()}")


+-----------+-----------+----+---------+--------+-------------+----------+--------------+----------+------------------+----------------+----------------+---------+
|customer_id|name       |age |city     |segment |annual_income|join_date |transaction_id|account_id|transaction_amount|transaction_type|transaction_date|branch_id|
+-----------+-----------+----+---------+--------+-------------+----------+--------------+----------+------------------+----------------+----------------+---------+
|C101       |Arun Sharma|34  |Mumbai   |Premium |850000       |2019-03-15|T001          |A1001     |5000.0            |Debit           |2024-01-15      |B01      |
|C101       |Arun Sharma|34  |Mumbai   |Premium |850000       |2019-03-15|T004          |A1001     |3000.0            |Debit           |2024-01-18      |B01      |
|C102       |Priya Sen  |28  |Delhi    |Standard|1200000      |2020-07-22|T002          |A1002     |15000.0           |Credit          |2024-01-16      |B02      |
|C102       |Pri

In [17]:
# Identify ALL data quality issues at once
data_issues = full_view.filter(
    col("transaction_id").isNull() |   # customers with no transactions
    col("name").isNull()               # transactions with no customer record
)

data_issues.select(
    "customer_id", "name",
    "transaction_id", "transaction_amount"
).show()

+-----------+-----------+--------------+------------------+
|customer_id|       name|transaction_id|transaction_amount|
+-----------+-----------+--------------+------------------+
|       C108|Deepa Menon|          NULL|              NULL|
|       C109|       NULL|          T009|           12000.0|
+-----------+-----------+--------------+------------------+



## Semi-join = inner joins + distinct 
- Returns only LEFT rows WHERE a match exists in right
- Returns ONLY left table's columns — no right table columns

In [19]:
# Left semi join — customers WHO HAVE at least one transaction
# No transaction columns in output — just confirmation of existence
active_customers = customers.join(
    transactions,
    on="customer_id",
    how="leftsemi"
)

active_customers.show()
print(f"Active customers: {active_customers.count()}")


+-----------+-----------+---+---------+--------+-------------+----------+
|customer_id|       name|age|     city| segment|annual_income| join_date|
+-----------+-----------+---+---------+--------+-------------+----------+
|       C101|Arun Sharma| 34|   Mumbai| Premium|       850000|2019-03-15|
|       C102|  Priya Sen| 28|    Delhi|Standard|      1200000|2020-07-22|
|       C103|  Rahul Das| 45|  Kolkata|Standard|       450000|2018-11-10|
|       C104| Meena Iyer| 52|  Chennai| Premium|       950000|2017-05-30|
|       C105| Suresh Roy| 23|     Pune|   Basic|       320000|2023-01-14|
|       C106|Fatima Khan| 39|Hyderabad| Premium|       780000|2021-09-05|
|       C107|Vikram Nair| 31|Bangalore|Standard|      1500000|2020-02-28|
+-----------+-----------+---+---------+--------+-------------+----------+

Active customers: 7


## LEFT ANTI JOIN: non-existence check 
- Returns LEFT rows WHERE NO match exists in right


In [20]:
# Left anti join — customers with NO transactions at all
dormant_customers = customers.join(
    transactions,
    on="customer_id",
    how="leftanti"
)

dormant_customers.show()

+-----------+-----------+---+-----+-------+-------------+----------+
|customer_id|       name|age| city|segment|annual_income| join_date|
+-----------+-----------+---+-----+-------+-------------+----------+
|       C108|Deepa Menon| 44|Kochi|  Basic|       290000|2022-06-18|
+-----------+-----------+---+-----+-------+-------------+----------+



In [21]:
# Find loans for customers NOT in our customer master 
orphaned_loans = loans.join(
    customers,
    on="customer_id",
    how="leftanti"
)
orphaned_loans.show()

+-----------+-------+-----------+-------------+-------------+---------+-----------------+
|customer_id|loan_id|loan_amount|    loan_type|interest_rate|   status|disbursement_date|
+-----------+-------+-----------+-------------+-------------+---------+-----------------+
|       C110|   L005|      80000|Personal Loan|         14.5|Defaulted|       2020-09-30|
+-----------+-------+-----------+-------------+-------------+---------+-----------------+



In [22]:
# Find customers with no credit score on file
no_credit_score = customers.join(
    credit_scores,
    on="customer_id",
    how="leftanti"
)
no_credit_score.select("customer_id", "name", "segment").show()

+-----------+-----------+-------+
|customer_id|       name|segment|
+-----------+-----------+-------+
|       C104| Meena Iyer|Premium|
|       C108|Deepa Menon|  Basic|
+-----------+-----------+-------+



## Cross join - it's like cross product every row of left df is mapped to every row in the right df

In [24]:
cross = customers.crossJoin(branches)
print(f"Customers: {customers.count()}")
print(f"Branches:  {branches.count()}")
print(f"Cross:     {cross.count()}") 
cross.show(5)

Customers: 8
Branches:  7
Cross:     56
+-----------+-----------+---+------+-------+-------------+----------+---------+---------------+-------+------+----------+
|customer_id|       name|age|  city|segment|annual_income| join_date|branch_id|    branch_name|   city|region|manager_id|
+-----------+-----------+---+------+-------+-------------+----------+---------+---------------+-------+------+----------+
|       C101|Arun Sharma| 34|Mumbai|Premium|       850000|2019-03-15|      B01|   Mumbai-North| Mumbai|  West|      M001|
|       C101|Arun Sharma| 34|Mumbai|Premium|       850000|2019-03-15|      B02|     Delhi-East|  Delhi| North|      M002|
|       C101|Arun Sharma| 34|Mumbai|Premium|       850000|2019-03-15|      B03|Kolkata-Central|Kolkata|  East|      M003|
|       C101|Arun Sharma| 34|Mumbai|Premium|       850000|2019-03-15|      B04|  Chennai-South|Chennai| South|      M004|
|       C101|Arun Sharma| 34|Mumbai|Premium|       850000|2019-03-15|      B05|      Pune-West|   Pune|  W

### Joining on multiple columns

In [26]:
# Join on multiple columns — more precise matching
result = customers.join(
    transactions.join(branches, on="branch_id", how="inner"),
    on=["customer_id"],
    how="inner"
)

result.show()

+-----------+-----------+---+---------+--------+-------------+----------+---------+--------------+----------+------------------+----------------+----------------+-----------------+---------+------+----------+
|customer_id|       name|age|     city| segment|annual_income| join_date|branch_id|transaction_id|account_id|transaction_amount|transaction_type|transaction_date|      branch_name|     city|region|manager_id|
+-----------+-----------+---+---------+--------+-------------+----------+---------+--------------+----------+------------------+----------------+----------------+-----------------+---------+------+----------+
|       C101|Arun Sharma| 34|   Mumbai| Premium|       850000|2019-03-15|      B01|          T001|     A1001|            5000.0|           Debit|      2024-01-15|     Mumbai-North|   Mumbai|  West|      M001|
|       C102|  Priya Sen| 28|    Delhi|Standard|      1200000|2020-07-22|      B02|          T002|     A1002|           15000.0|          Credit|      2024-01-16|  

## Duplicate column issue

In [ ]:
result = transactions.join(customers, on="customer_id", how="inner")
result.printSchema()

## here we don't get the duplicate column issue as we are using inner join and on

root
 |-- customer_id: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- annual_income: integer (nullable = true)
 |-- join_date: date (nullable = true)



In [29]:
## when we use the column expression join we get the ambiguity
result = transactions.join(
    customers,
    transactions.customer_id == customers.customer_id,
    "inner"
)
result.printSchema()

## customer_id appears twice, which causes problem and confusion

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- annual_income: integer (nullable = true)
 |-- join_date: date (nullable = true)



In [30]:
result.columns

['transaction_id',
 'customer_id',
 'account_id',
 'transaction_amount',
 'transaction_type',
 'transaction_date',
 'branch_id',
 'customer_id',
 'name',
 'age',
 'city',
 'segment',
 'annual_income',
 'join_date']

## Solutions

In [31]:
# Solution 1: Rename before joining
customers_renamed = customers.withColumnRenamed("customer_id", "cust_id")
result = transactions.join(customers_renamed,
    transactions.customer_id == customers_renamed.cust_id, "inner")

result.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- cust_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- annual_income: integer (nullable = true)
 |-- join_date: date (nullable = true)



In [32]:
# Solution 2: Select and alias only what we need after the join
result = transactions.join(customers,
    transactions.customer_id == customers.customer_id, "inner"
).select(
    transactions.customer_id,         # explicitly pick which one
    transactions.transaction_amount,
    customers.name,
    customers.city
)

result.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)



In [33]:
# Solution 3: Drop the duplicate after join
result = transactions.join(
    customers,
    transactions.customer_id == customers.customer_id,
    "inner"
).drop(customers.customer_id)      

result.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- annual_income: integer (nullable = true)
 |-- join_date: date (nullable = true)



### Chain of joins 

In [35]:
enriched = transactions \
    .join(customers,
          on="customer_id", how="left") \
    .join(branches,
          on="branch_id",   how="left") \
    .join(credit_scores,
          on="customer_id", how="left") \
    .join(loans,
          on="customer_id", how="left") \
    .select(
        # Transaction details
        col("transaction_id"),
        col("transaction_amount"),
        col("transaction_type"),
        col("transaction_date"),

        # Customer details
        col("customer_id"),
        customers.name.alias("customer_name"),
        col("segment"),
        col("annual_income"),

        # Branch details
        col("branch_id"),
        branches.city.alias("branch_city"),
        col("region"),

        # Credit info
        col("credit_score"),
        col("bureau_name"),

        # Loan info
        col("loan_id"),
        col("loan_type"),
        col("status").alias("loan_status")
    )

enriched.show(truncate=False)
enriched.printSchema()

+--------------+------------------+----------------+----------------+-----------+-------------+--------+-------------+---------+-----------+------+------------+-----------+-------+-------------+-----------+
|transaction_id|transaction_amount|transaction_type|transaction_date|customer_id|customer_name|segment |annual_income|branch_id|branch_city|region|credit_score|bureau_name|loan_id|loan_type    |loan_status|
+--------------+------------------+----------------+----------------+-----------+-------------+--------+-------------+---------+-----------+------+------------+-----------+-------+-------------+-----------+
|T001          |5000.0            |Debit           |2024-01-15      |C101       |Arun Sharma  |Premium |850000       |B01      |Mumbai     |West  |720         |CIBIL      |L001   |Home Loan    |Active     |
|T002          |15000.0           |Credit          |2024-01-16      |C102       |Priya Sen    |Standard|1200000      |B02      |Delhi      |North |680         |CIBIL      |

In [ ]:
## Union is used to stack rows instead of columns (similar to concat in pandas)

## example
# Scenario: transactions from two different months arrive as separate files
jan_transactions = spark.read.csv("jan_transactions.csv", header=True, inferSchema=True)
feb_transactions = spark.read.csv("feb_transactions.csv", header=True, inferSchema=True)

# Union — stack rows, schemas must match exactly (same columns, same order)
all_transactions = jan_transactions.union(feb_transactions)

union() — stacks rows. Both DataFrames must have the same number of columns in the same order. Column names don't matter — it matches by position.

In [ ]:
# unionByName — safer, matches by column name not position
all_transactions = jan_transactions.unionByName(feb_transactions)

unionByName() — matches columns by name, not position. Use this always — it's safer when column order might differ between files.

In [ ]:
# Remove duplicates after union (in case same transaction appears in both files)
all_transactions = jan_transactions.unionByName(feb_transactions).distinct()

In [ ]:
# unionByName with mismatched schemas
# One file might have a column the other doesn't
all_transactions = jan_transactions.unionByName(
    feb_transactions,
    allowMissingColumns=True   # missing columns filled with null
)

## SQL Join & Combine Cheat Sheet (Banking Context)


- **"Show me X with details from Y where both exist"**  
  → `INNER JOIN`  
  _Example:_ Transactions with customer profiles (only known customers)

- **"Show me ALL of X, add Y details where available"**  
  → `LEFT JOIN` ← most common in banking  
  _Example:_ All transactions enriched with optional credit score

- **"Show me ALL of Y, add X details where available"**  
  → `RIGHT JOIN` (or swap tables and use LEFT JOIN)  
  _Example:_ All branches, even those with no transactions today

- **"Show me EVERYTHING from both, regardless of match"**  
  → `FULL OUTER JOIN`  
  _Example:_ Reconciliation — find gaps in both systems

- **"Does X exist in Y? (yes/no, no Y columns needed)"**  
  → `LEFT SEMI JOIN`  
  _Example:_ Which customers have at least one loan?

- **"What's in X that's NOT in Y?"**  
  → `LEFT ANTI JOIN`  
  _Example:_ Customers with no credit score on file

- **"Combine all rows vertically (same structure)"**  
  → `UNION / unionByName`  
  _Example:_ Jan + Feb + Mar transactions into one table

- **"Every combination of X and Y"**  
  → `CROSS JOIN` (use with extreme caution)  
  _Example:_ All customers × all product scenarios

### For joining one small table with one massive table we use broadcast


In [ ]:
# WITH broadcast — branches (7 rows!) is copied to every worker
# No shuffle needed for the large table at all
transactions.join(broadcast(branches), on="branch_id", how="left")

`broadcast(small_df) — wraps the small DataFrame and tells Spark to copy it to every worker node. Each worker can then do the join locally without any network communication. For small reference tables (branches, products, account types, region codes) this can make a join 10–100x faster.`